In [63]:
import os
from operator import index

import rasterio
import rioxarray as rxr
import glob
import geopandas as gpd
import pandas as pd

In [9]:
dir1 = "C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1"
dir2 = "C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2"

task1 = []
task2 = []
for root, dirs, files in os.walk(dir1):
    for directory in dirs:
        task1.append(os.path.join(root, directory))
        
for root, dirs, files in os.walk(dir2):
    for directory in dirs:
        task2.append(os.path.join(root, directory))
print(task1)
print("-------------------------------")
print(task2)

['C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20230405', 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20240315', 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20240418', 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20250404', 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20250501']
-------------------------------
['C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20230405', 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20240315', 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20240418', 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20250404', 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20250501']


In [56]:
def swe_units(task_dir):
    for root, dirs, files in os.walk(task_dir):
        for directory in dirs:
            date = os.path.basename(directory)

            if os.path.isfile(glob.glob(os.path.join(root,directory, "*EB_swe*"))[0]):
                with rasterio.open(glob.glob(os.path.join(root,directory, "*EB_swe*"))[0]) as src:
                    data = src.read(1, masked=True)
                    profile = src.profile
                    profile.update(dtype=rasterio.float32, nodata=-9999)
                    outfile = os.path.join(root,directory, f"{date}_EB_cm.tif")
                    EB_cm = data * 2.54
                with rasterio.open(outfile, "w", **profile) as dst:
                    dst.write(EB_cm.astype("float32"), 1)
            else:
                print("No energy balance raster found")
                
            if os.path.isfile(glob.glob(os.path.join(root,directory, "*TI_swe*"))[0]):
                with rasterio.open(glob.glob(os.path.join(root,directory, "*TI_swe*"))[0]) as src:
                    data = src.read(1, masked=True)
                    profile = src.profile
                    profile.update(dtype=rasterio.float32, nodata=-9999)
                    outfile = os.path.join(root,directory, f"{date}_TI_cm.tif")
                    TI_cm = data * 2.54
                with rasterio.open(outfile, "w", **profile) as dst:
                    dst.write(TI_cm.astype("float32"), 1)
            else:
                print("No temperature index raster found") 
            
            if os.path.isfile(glob.glob(os.path.join(root,directory, "*swed*"))[0]):
                with rasterio.open(glob.glob(os.path.join(root,directory, "*swed*"))[0]) as src:
                    data = src.read(1, masked=True)
                    profile = src.profile
                    profile.update(dtype=rasterio.float32, nodata=-9999)
                    outfile = os.path.join(root,directory, f"{date}_SnowModel_cm.tif")
                    SM_cm = data * 100
                with rasterio.open(outfile, "w", **profile) as dst:
                    dst.write(SM_cm.astype("float32"), 1)
            else:
                print("No SnowModel raster found") 
            
            if os.path.isfile(glob.glob(os.path.join(root,directory, "*specific_mass*"))[0]):
                with rasterio.open(glob.glob(os.path.join(root,directory, "*specific_mass*"))[0]) as src:
                    data = src.read(1, masked=True)
                    profile = src.profile
                    profile.update(dtype=rasterio.float32, nodata=-9999)
                    outfile = os.path.join(root,directory, f"{date}_iSnobal_cm.tif")
                    iSnobal_cm = data / 10
                with rasterio.open(outfile, "w", **profile) as dst:
                    dst.write(iSnobal_cm.astype("float32"), 1)
            else:
                print("No iSnobal raster found") 
                
                
                        

In [58]:
#swe_units("C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2")
swe_units("C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1")

In [72]:
def task_process(task_dir):
    swedata = {}
    for root, dirs, files in os.walk(task_dir):
        for directory in dirs:
            date = os.path.basename(directory)
            swedata[date] = {}
            swedata[date]["Energy Balance"] = glob.glob(os.path.join(root,directory, "*EB_cm*"))[0]
            swedata[date]["Temperature Index"] = glob.glob(os.path.join(root,directory, "*TI_cm*"))[0]
            swedata[date]["SnowModel"] = glob.glob(os.path.join(root,directory, "*SnowModel_cm*"))[0]
            swedata[date]["iSnobal"] = glob.glob(os.path.join(root,directory, "*iSnobal_cm*"))[0]
    return swedata

In [73]:
task2 = task_process("C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2")
task1 = task_process("C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1")
print(task2)
print("----------------------------------------------")
print(task1)

{'20230405': {'Energy Balance': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20230405\\20230405_EB_cm.tif', 'Temperature Index': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20230405\\20230405_TI_cm.tif', 'SnowModel': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20230405\\20230405_SnowModel_cm.tif', 'iSnobal': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20230405\\20230405_iSnobal_cm.tif'}, '20240315': {'Energy Balance': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20240315\\20240315_EB_cm.tif', 'Temperature Index': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20240315\\20240315_TI_cm.tif', 'SnowModel': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20240315\\20240315_SnowModel_cm.tif', 'iSnobal': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20240315\\20240315_iSnobal_cm.tif'}, '20240418': {'Energy Balance': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20240418\\20240418_EB_cm.tif'

In [74]:
for date, models in task1.items():
    
    # Loop through each model for the current date
        for model_name, raster in models.items():
            print(raster)

C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20230405\20230405_EB_cm.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20230405\20230405_TI_cm.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20230405\20230405_SnowModel_cm.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20230405\20230405_iSnobal_cm.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240315\20240315_EB_cm.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240315\20240315_TI_cm.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240315\20240315_SnowModel_cm.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240315\20240315_iSnobal_cm.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240418\20240418_EB_cm.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240418\20240418_TI_cm.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240418\20240418_SnowModel_cm.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\2

In [97]:
def calculate_total_weighted_sum(dictionary,  csv_file_path, tasknumber):
    """
    Calculates the sum of all raster pixel values, weighted by the pixel area.

    Args:
        filename (str): The path to the input raster file.

    Returns:
        float: The total weighted sum (e.g., volume if pixel values are height in meters).
    """
    
    results_list = []
    
    for date, models in dictionary.items():
    
    # Loop through each model for the current date
        for model_name, raster in models.items():
    
            with rasterio.open(raster) as src:
                # Read the first band as a masked numpy array to handle NoData values
                array = src.read(1, masked=True)
                non_na_count_masked = array.count()
                # Get metadata for pixel size (resolution)
                # Assumes the CRS units are appropriate for area calculation (e.g., meters)
                pixel_width = src.res[0]
                pixel_height = src.res[1]
                pixel_area = abs(pixel_width * pixel_height) # Use abs() as y resolution is often negative
        
       
                # Multiply each pixel value by the pixel area (element-wise multiplication with numpy)
                swe_m = array / 100
                weighted_volumes = swe_m * pixel_area
        
                # Sum all the weighted values (only valid pixels are summed due to masking)
                total_sum = weighted_volumes.sum()
                
                row = {
                    "task_number": tasknumber,
                    "date": date,
                    "pixel width (m)": pixel_width,
                    "pixel height (m)": pixel_height,
                    "pixel area (m2)": pixel_area,
                    "pixel count": non_na_count_masked,
                    "total area": pixel_area * non_na_count_masked,
                    "model name": model_name,
                    "total swe (m3)": total_sum
                }
                
                results_list.append(row)
                    
    df = pd.DataFrame(results_list)
    df.to_csv(csv_file_path, index=False)
        


In [100]:
calculate_total_weighted_sum(task2, "C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2/Task2_totalSWE.csv", 2)
calculate_total_weighted_sum(task1, "C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1/Task1_totalSWE.csv", 1)